In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
import sys

sys.path.insert(1, "../../../")

In [ ]:
import logging
import os

from pydantic import SecretStr

from benchmark_qed.autod.data_processor.embedding import TextEmbedder
from benchmark_qed.autod.data_processor.text_splitting import TokenTextSplitter
from benchmark_qed.autod.io.document import (
    create_documents,
    save_documents,
)
from benchmark_qed.autod.io.text_unit import create_text_units, save_text_units
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.llm.factory import ModelFactory

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.ERROR)

In [ ]:
%load_ext dotenv
%dotenv

# Experiment 11d: MMR-Based Text Sampling

This experiment demonstrates the **Maximal Marginal Relevance (MMR)** sampler for selecting diverse and representative text units from a corpus. MMR balances two competing objectives:

- **Quality**: selecting text units that are most representative of the corpus
- **Diversity**: avoiding redundant or near-duplicate selections

The MMR score for a candidate text unit `q` is computed as:

```
MMR(q) = λ * quality(q) - (1 - λ) * max_similarity(q, selected)
```

Where `λ` (lambda) controls the quality-diversity trade-off:
- `λ = 1.0` → pure quality maximization (select most representative items)
- `λ = 0.0` → pure diversity maximization (select most dissimilar items)
- `λ = 0.5` → balanced quality and diversity (default)

This experiment explores how varying `λ` affects the sample composition and compares MMR sampling against KMeans-based sampling.

## Configs

In [ ]:
INPUT_DATA_PATH = "../../datasets/AP_news/raw_data"
OUTPUT_DATA_PATH = "./output/AP_news/processed_data"
TEXT_COLUMN = "body_nitf"
METADATA_COLUMNS = ["headline", "firstcreated"]
JSON_ENCODING = "utf-8-sig"

# tokenizer used for chunking documents into text units
ENCODING_MODEL = "o200k_base"
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

# llm/embedding settings
API_KEY = SecretStr(os.getenv("OPENAI_API_KEY", ""))
EMBEDDING_MODEL = "text-embedding-3-large"

# sampling settings
SAMPLE_SIZE = 500  # total number of text units to sample

## Step 1: Load Documents

In [ ]:
documents = create_documents(
    input_path=INPUT_DATA_PATH,
    input_type="json",
    text_tag=TEXT_COLUMN,
    metadata_tags=METADATA_COLUMNS,
    encoding=JSON_ENCODING,
)
document_df = save_documents(documents, OUTPUT_DATA_PATH)
print(f"Document count: {len(document_df)}")
document_df.head()

## Step 2: Create and Embed Text Units

Chunk documents into text units and generate embeddings. Embeddings are required by the MMR sampler to compute pairwise similarities between text units.

In [ ]:
text_splitter = TokenTextSplitter(
    encoding_name=ENCODING_MODEL,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

text_embedder = TextEmbedder(
    ModelFactory.create_embedding_model(
        LLMConfig(
            model=EMBEDDING_MODEL,
            api_key=API_KEY,
            llm_provider=LLMProvider.OpenAIEmbedding,
        )
    )
)

text_units = await create_text_units(
    documents=documents,
    metadata_tags=METADATA_COLUMNS,
    text_splitter=text_splitter,
    text_embedder=text_embedder,
    embed_text=True,
)
text_unit_df = save_text_units(text_units, OUTPUT_DATA_PATH)
print(f"Text unit count: {len(text_unit_df)}")
text_unit_df.head()

## Step 3: MMR Sampling

Apply the MMR sampler with different `lambda` values to observe the quality-diversity trade-off.

In [ ]:
from benchmark_qed.autod.sampler.sampling.mmr_sampler import MMRTextSampler

### 3a. Balanced MMR (λ = 0.5) — default

In [ ]:
mmr_balanced = MMRTextSampler(lambda_param=0.5)
sample_balanced = mmr_balanced.sample(
    text_units=text_units,
    sample_size=SAMPLE_SIZE,
)
print(f"Balanced MMR (λ=0.5) sample size: {len(sample_balanced)}")

### 3b. Diversity-Focused MMR (λ = 0.0)

In [ ]:
mmr_diverse = MMRTextSampler(lambda_param=0.0)
sample_diverse = mmr_diverse.sample(
    text_units=text_units,
    sample_size=SAMPLE_SIZE,
)
print(f"Diversity-focused MMR (λ=0.0) sample size: {len(sample_diverse)}")

### 3c. Quality-Focused MMR (λ = 1.0)

In [ ]:
mmr_quality = MMRTextSampler(lambda_param=1.0)
sample_quality = mmr_quality.sample(
    text_units=text_units,
    sample_size=SAMPLE_SIZE,
)
print(f"Quality-focused MMR (λ=1.0) sample size: {len(sample_quality)}")

## Step 4: Evaluate Sampling Diversity

Measure the intra-sample diversity for each configuration by computing the average pairwise cosine distance between selected text unit embeddings. Higher average distance indicates greater diversity.

In [ ]:
import numpy as np


def mean_pairwise_distance(samples: list) -> float:
    """Compute mean pairwise cosine distance for a list of text units."""
    embeddings = np.array([u.text_embedding for u in samples if u.text_embedding is not None])
    if len(embeddings) < 2:
        return 0.0
    # Normalize embeddings
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    embeddings_norm = embeddings / norms
    # Cosine similarity matrix
    sim_matrix = embeddings_norm @ embeddings_norm.T
    # Extract upper triangle (exclude diagonal)
    upper = sim_matrix[np.triu_indices(len(embeddings_norm), k=1)]
    # Distance = 1 - similarity
    return float(np.mean(1.0 - upper))


diversity_balanced = mean_pairwise_distance(sample_balanced)
diversity_diverse = mean_pairwise_distance(sample_diverse)
diversity_quality = mean_pairwise_distance(sample_quality)

print("Mean pairwise cosine distance (higher = more diverse):")
print(f"  MMR λ=0.5 (balanced):   {diversity_balanced:.4f}")
print(f"  MMR λ=0.0 (diversity):  {diversity_diverse:.4f}")
print(f"  MMR λ=1.0 (quality):    {diversity_quality:.4f}")

## Step 5: Comparison with KMeans Sampling

Compare the MMR sampler (λ=0.5) against the KMeans-based sampler to understand the diversity characteristics of each approach.

In [ ]:
from benchmark_qed.autod.sampler.enums import ClusterRepresentativeSelectionType
from benchmark_qed.autod.sampler.sampling.kmeans_sampler import KmeansTextSampler

NUM_CLUSTERS = 50
NUM_SAMPLES_PER_CLUSTER = 10

kmeans_sampler = KmeansTextSampler()
sample_kmeans = kmeans_sampler.sample(
    text_units=text_units,
    sample_size=None,
    num_clusters=NUM_CLUSTERS,
    num_samples_per_cluster=NUM_SAMPLES_PER_CLUSTER,
    cluster_representative_selection_type=ClusterRepresentativeSelectionType.CENTROID,
)
print(f"KMeans sample size: {len(sample_kmeans)}")

diversity_kmeans = mean_pairwise_distance(sample_kmeans)
print(f"KMeans mean pairwise cosine distance: {diversity_kmeans:.4f}")

In [ ]:
print("\nSampling diversity comparison:")
print(f"  KMeans ({NUM_CLUSTERS} clusters × {NUM_SAMPLES_PER_CLUSTER} samples): {diversity_kmeans:.4f}")
print(f"  MMR λ=0.5 (balanced):                                  {diversity_balanced:.4f}")
print(f"  MMR λ=0.0 (diversity):                                 {diversity_diverse:.4f}")
print(f"  MMR λ=1.0 (quality):                                   {diversity_quality:.4f}")

## Summary

This experiment compared the MMR sampler across different `lambda` configurations and against the KMeans sampler:

| Sampler | λ | Strategy | Expected Diversity |
|---------|---|----------|-------------------|
| MMR | 0.0 | Maximize diversity | Highest |
| MMR | 0.5 | Balanced (default) | Medium-High |
| MMR | 1.0 | Maximize quality | Lower |
| KMeans | N/A | Cluster-based | Medium |

**Key observations:**
- MMR with `λ=0.0` prioritizes coverage of the embedding space, yielding the most diverse sample.
- MMR with `λ=1.0` repeatedly selects centroid-proximal items, reducing diversity.
- The balanced setting (`λ=0.5`) provides a practical trade-off between representativeness and diversity.
- KMeans sampling achieves structured diversity by ensuring coverage across semantic clusters.

**Recommendation:** Use MMR with `λ=0.5` when you want a diverse sample without the need to specify the number of clusters upfront. Use KMeans when you need explicit control over the number of topic areas (clusters) covered in the sample.